# Export the IVF-PQ index for the Hugging Face Space

**Runtime:** standard CPU. Needs the outputs of `01_embed` and `02_index_bench` on Drive.

`02_index_bench` frees every index after measuring it, so this notebook rebuilds IVF-PQ the same way:
- the same coarse centroids (`results/ivf_centroids.npy`);
- the same 200,000-vector training sample (same seed);
- the same nlist and PQ size.

It then checks the rebuilt index against the saved results and writes the files the Space needs to
`space_assets/` on Drive:

| File | Content |
|---|---|
| `ivfpq.faissindex` | the IVF-PQ index (about 106 MB) |
| `db_labels.npy` | the class of every database image, in global-id order |
| `assets.json` | dataset files, shard offsets, model name, default nprobe, measured metrics |
| `class_names.json` | the 1,000 class names |

The last cell uploads them to a **private Hugging Face dataset repository**. Set `LAB2_HF_ASSETS_REPO`
(for example `your-username/imagenet-clip-ivfpq`) and add an `HF_TOKEN` with **write** access in
Colab's Secrets panel. The token is read from there and never printed.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu==1.11.0"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

## Configuration

In [ ]:
def env_str(name, default):
    return os.environ.get(name, default)


def env_int(name, default):
    return int(os.environ.get(name, default))


LAB2_ROOT = env_str("LAB2_ROOT", "/content/drive/MyDrive/cap6938-hw2")
LAB2_SEED = env_int("LAB2_SEED", 42)                                    # must match 02_index_bench
LAB2_NLIST = env_int("LAB2_NLIST", 4096)                                # must match 02_index_bench
LAB2_PQ_M = env_int("LAB2_PQ_M", 64)                                    # must match 02_index_bench
LAB2_TRAIN_SAMPLE = env_int("LAB2_TRAIN_SAMPLE", 200000)                # must match 02_index_bench
LAB2_TOPK = env_int("LAB2_TOPK", 10)
LAB2_OPERATING_NPROBE = env_int("LAB2_OPERATING_NPROBE", 16)            # default nprobe in the Space
LAB2_THREADS = env_int("LAB2_THREADS", os.cpu_count() or 1)
LAB2_DATASET = env_str("LAB2_DATASET", "benjamin-paine/imagenet-1k-256x256")
LAB2_NUM_VAL_FILES = env_int("LAB2_NUM_VAL_FILES", 2)
LAB2_HF_ASSETS_REPO = env_str("LAB2_HF_ASSETS_REPO", "")                # e.g. "your-username/imagenet-clip-ivfpq"; empty skips upload
LAB2_HF_ASSETS_PRIVATE = env_int("LAB2_HF_ASSETS_PRIVATE", 1)           # 1 creates the repository as private

EMB_DIR, RES_DIR = f"{LAB2_ROOT}/embeddings", f"{LAB2_ROOT}/results"
OUT_DIR = f"{LAB2_ROOT}/space_assets"
os.makedirs(OUT_DIR, exist_ok=True)

## Load embeddings metadata, labels and queries

In [ ]:
import gc, json, time
import numpy as np
import pandas as pd
import faiss

with open(f"{EMB_DIR}/manifest.json") as f:
    manifest = json.load(f)
assert "total_rows" in manifest, "run the checks cell at the end of 01_embed.ipynb first"
SHARDS = sorted(int(k) for k in manifest["shards"])
OFFSETS = {i: manifest["shards"][str(i)]["offset"] for i in SHARDS}
ROWS = {i: manifest["shards"][str(i)]["rows"] for i in SHARDS}
N, D = manifest["total_rows"], manifest["dim"]
NUM_TRAIN_FILES = len(SHARDS)

db_labels = np.concatenate([np.load(f"{EMB_DIR}/train_{i:02d}_labels.npy") for i in SHARDS]).astype(np.int16)
Q = np.ascontiguousarray(np.load(f"{EMB_DIR}/queries.npy"), dtype=np.float32)
q_labels = np.load(f"{EMB_DIR}/queries_labels.npy").astype(np.int32)
faiss.omp_set_num_threads(LAB2_THREADS)
print(f"database {N:,} x {D} | queries {len(Q):,} | faiss {faiss.__version__} | threads {faiss.omp_get_max_threads()}")

## Rebuild IVF-PQ

This is the same procedure as `02_index_bench`. The PQ codebooks are trained on the residuals of
the same 200,000-vector sample, then all 1.28M vectors are added one shard at a time.

In [ ]:
def training_sample():
    rng = np.random.default_rng(LAB2_SEED)
    ids = np.sort(rng.choice(N, size=min(LAB2_TRAIN_SAMPLE, N), replace=False))
    out, pos = np.empty((len(ids), D), np.float32), 0
    for i in SHARDS:
        sel = ids[(ids >= OFFSETS[i]) & (ids < OFFSETS[i] + ROWS[i])] - OFFSETS[i]
        x = np.load(f"{EMB_DIR}/train_{i:02d}.npy", mmap_mode="r")
        out[pos:pos + len(sel)] = x[sel]
        pos += len(sel)
    return out


quantizer = faiss.IndexFlatIP(D)
quantizer.add(np.load(f"{RES_DIR}/ivf_centroids.npy"))
index = faiss.IndexIVFPQ(quantizer, D, LAB2_NLIST, LAB2_PQ_M, 8, faiss.METRIC_INNER_PRODUCT)

t0 = time.perf_counter()
xt = training_sample()
index.train(xt)          # the quantizer already holds nlist centroids, so only PQ is trained
del xt
gc.collect()
train_s = time.perf_counter() - t0
t0 = time.perf_counter()
for i in SHARDS:
    index.add(np.ascontiguousarray(np.load(f"{EMB_DIR}/train_{i:02d}.npy", mmap_mode="r"), dtype=np.float32))
add_s = time.perf_counter() - t0
assert index.ntotal == N
print(f"trained in {train_s:.0f}s, added {N:,} vectors in {add_s:.0f}s")

## Check the rebuilt index against the saved results

Precision@10 and recall@10 at the operating nprobe should match the `ivf_pq64` row in
`results.csv`. Small differences are possible because k-means can sum in a different order on a
different number of threads.

In [ ]:
index.nprobe = LAB2_OPERATING_NPROBE
_, ids = index.search(Q, LAB2_TOPK)
gt = np.load(f"{RES_DIR}/gt_flat_top{LAB2_TOPK}.npy")
p10 = (db_labels[ids].astype(np.int32) == q_labels[:, None]).mean()
r10 = ((ids[:, :, None] == gt[:, None, :]).any(axis=2).sum(axis=1) / LAB2_TOPK).mean()

res = pd.read_csv(f"{RES_DIR}/results.csv")
ref = res[res["tag"] == f"ivf_pq64_nprobe{LAB2_OPERATING_NPROBE}"].iloc[0]
saved = np.load(f"{RES_DIR}/topk_ivf_pq64_nprobe{LAB2_OPERATING_NPROBE}.npy")
same = (np.sort(ids, axis=1) == np.sort(saved, axis=1)).all(axis=1).mean()
print(f"rebuilt: P@10 {p10:.4f}, R@10 {r10:.4f}")
print(f"saved:   P@10 {ref['p_at_10']:.4f}, R@10 {ref['recall_at_10']:.4f}")
print(f"queries with an identical top {LAB2_TOPK}: {same:.1%}")
assert abs(p10 - ref["p_at_10"]) < 0.01 and abs(r10 - ref["recall_at_10"]) < 0.01, \
    "rebuilt index does not reproduce the reported metrics"
print("CHECK PASSED")

## Save the Space assets to Drive

In [ ]:
def save_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


faiss.write_index(index, f"{OUT_DIR}/ivfpq.faissindex")
np.save(f"{OUT_DIR}/db_labels.npy", db_labels)
with open(f"{EMB_DIR}/class_names.json") as f:
    save_json(f"{OUT_DIR}/class_names.json", json.load(f))
save_json(f"{OUT_DIR}/assets.json", {
    "dataset": manifest.get("dataset", LAB2_DATASET),
    "train_files": [f"data/train-{i:05d}-of-{NUM_TRAIN_FILES:05d}.parquet" for i in SHARDS],
    "train_offsets": [OFFSETS[i] for i in SHARDS],
    "val_files": [f"data/validation-{i:05d}-of-{LAB2_NUM_VAL_FILES:05d}.parquet"
                  for i in range(LAB2_NUM_VAL_FILES)],
    "total_rows": N, "dim": D,
    "model": manifest["model"], "pretrained": manifest["pretrained"],
    "index": {"type": f"IVF{LAB2_NLIST},PQ{LAB2_PQ_M}", "metric": "inner_product",
              "default_nprobe": LAB2_OPERATING_NPROBE},
    "measured": {"p_at_10": round(float(p10), 4), "recall_at_10": round(float(r10), 4),
                 "lat_median_ms": round(float(ref["lat_median_ms"]), 3),
                 "bytes": int(os.path.getsize(f"{OUT_DIR}/ivfpq.faissindex"))},
})
for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:20s} {os.path.getsize(f'{OUT_DIR}/{name}') / 1e6:8.1f} MB")

## Upload to a private Hugging Face dataset repository

This is skipped when `LAB2_HF_ASSETS_REPO` is empty. The token comes from Colab Secrets (`HF_TOKEN`)
or the environment. The Space reads the repository with its own `HF_TOKEN` secret, which needs
**read** access only.

In [ ]:
if not LAB2_HF_ASSETS_REPO:
    print("LAB2_HF_ASSETS_REPO is empty: upload skipped. The assets are in", OUT_DIR)
else:
    from huggingface_hub import HfApi

    token = os.environ.get("HF_TOKEN")
    if token is None and IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    assert token, "add an HF_TOKEN with write access in Colab Secrets"
    api = HfApi(token=token)
    api.create_repo(LAB2_HF_ASSETS_REPO, repo_type="dataset", private=bool(LAB2_HF_ASSETS_PRIVATE),
                    exist_ok=True)
    api.upload_folder(repo_id=LAB2_HF_ASSETS_REPO, repo_type="dataset", folder_path=OUT_DIR,
                      commit_message="Upload IVF-PQ index and metadata for the image search Space")
    print(f"uploaded to https://huggingface.co/datasets/{LAB2_HF_ASSETS_REPO}")